In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport

c:\Users\Asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Asus\AppData\Local\Temp\ipykernel_3708\2963060773.py:5: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [11]:
import sys
from pathlib import Path
sys.path.insert(0, str(next(p for p in (Path.cwd(), *Path.cwd().parents)
                            if (p / "scripts" / "project.py").exists()) / "scripts"))
from project import *  # ROOT, CACHE_FILE, LEAD_SET, MODELS_DIR, RESULTS_DIR, load_cache, ...
cache = np.load(CACHE_FILE, allow_pickle=True)
X_feat_train = cache["X_feat_train"]
y_train = cache["y_train"]
class_names = cache["class_names"]


In [12]:
feature_names = [
    "Mean_HR", "Min_HR", "Max_HR", "HR_Std", 
    "Mean_RR", "RR_Std", "RR_Range", "RR_CV",
    "SDNN", "RMSSD", "pNN50", "Sample_Entropy", 
    "Mean_PR", "Mean_QRS", "Mean_QT", "Mean_QTc",
    "SNR", "Baseline_Wander"
] 

In [13]:
df_features = pd.DataFrame(X_feat_train[:, :len(feature_names)], columns=feature_names)
df_targets = pd.DataFrame(y_train, columns=class_names)
df_combined = pd.concat([df_features, df_targets], axis=1)


In [14]:
# 3. Generate ydata-profiler HTML Report
print("Generating comprehensive ydata-profiling report...")
profile = ProfileReport(df_combined, title="ECG Feature-Target Correlation Profiling", explorative=True)
profile.to_file(FEATURE_RESULTS_DIR / "ecg_feature_profiling_report.html")
print("Saved report to 'ecg_feature_profiling_report.html'")

Generating comprehensive ydata-profiling report...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 17.77it/s]

Saved report to 'ecg_feature_profiling_report.html'


In [15]:
plt.figure(figsize=(14, 8))
# Calculate Pearson correlation
corr_matrix = df_combined.corr()

# Isolate just the correlation between the features (rows) and the 5 superclass targets (columns)
target_corr = corr_matrix.loc[feature_names, class_names]

In [16]:
sns.heatmap(target_corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Correlation: Handcrafted ECG Features vs. Diagnostic Superclasses", fontsize=14, fontweight='bold')
plt.xlabel("Target Ailments (Superclasses)", fontsize=12)
plt.ylabel("Extracted Numerical Features", fontsize=12)
plt.tight_layout()
plt.savefig(FEATURE_RESULTS_DIR / "feature_target_correlation_heatmap.png", dpi=300)
plt.show()

C:\Users\Asus\AppData\Local\Temp\ipykernel_3708\1366304594.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
